# Эксперименты

Весь код лежит в модулях проекта, здесь только вызовы. Ноутбук запускается из корня проекта.

| Раздел | Что делает | Нужен GPU |
|---|---|---|
| 0. Настройка | импорты, пути | нет |
| 1. Данные | чтение `.jsonl`, загрузчики | нет |
| 2. Обучение | LoRA 0.5B (два набора слоёв) и QLoRA 1.5B; адаптеры сохраняются в `models/` | да (пропустите, если адаптеры уже есть) |
| 3. Оценка | загрузка готовых адаптеров, прогон по всем тестам, результаты в `results/` | желательно |
| 4. Сводка | метрики по сохранённым результатам | нет |
| 5. Разбор ошибок | какие поля ошибаются, примеры | нет |
| 6. Забывание | ответы на обычные вопросы с адаптером и без | желательно |
| 7. Свой текст | один пример руками | желательно |
| 8. Деплой | слияние адаптера, GGUF, квантизация, Ollama, оценка | нет (нужны `llama.cpp` и Ollama) |

QLoRA обучается только на NVIDIA GPU (нужен `bitsandbytes`), остальное работает и на Mac (`mps`) и на CPU.

## 0. Настройка

In [ ]:
import glob
import os
import torch

from config import qlora_model_name
from data_utils import load_jsonl, make_loader
from models import make_model, make_qlora_model, load_lora_model, load_qlora_model, free_memory
from training import train
from evaluation import run_all, metrics, load_results, field_errors
from inference import chat, generate_json
from deploy import export_gguf, ollama_generate

ATTENTION = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
ATTENTION_MLP = ATTENTION + ['gate_proj', 'up_proj', 'down_proj']

# итоговые адаптеры (обучение сохраняет их в save_dir, чекпойнты эпох лежат в подпапках epoch1, epoch2)
ADAPTERS = {
    'attention_0.5B': 'models/attention',
    'attention_mlp_0.5B': 'models/attention_mlp',
    'qlora_1.5B': 'models/qlora',
}
RESULTS_DIR = 'results'

## 1. Данные

In [ ]:
train_set = load_jsonl('data/train.jsonl')
val_set = load_jsonl('data/val.jsonl')

loader = make_loader(train_set, shuffle=True)
val_loader = make_loader(val_set, shuffle=False)

TESTS = {
    'synth_test': load_jsonl('data/test.jsonl'),
    'real_1': load_jsonl('data/rent_msk_test.jsonl'),
    'real_2': load_jsonl('data/rent_msk_test_v2_diverse.jsonl'),
}

print('train', len(train_set), '| val', len(val_set), '| тесты', {k: len(v) for k, v in TESTS.items()})

## 2. Обучение

Пропустите этот раздел, если адаптеры уже сохранены. После обучения модель удаляется из памяти, оценка загружает её заново из файла.

### 2.1. LoRA на 0.5B, только внимание

In [ ]:
model = make_model(ATTENTION)
model.print_trainable_parameters()

losses = train(model, loader, epochs=2, val_loader=val_loader, save_dir=ADAPTERS['attention_0.5B'])

del model
free_memory()

### 2.2. LoRA на 0.5B, внимание + MLP

In [ ]:
model = make_model(ATTENTION_MLP)
model.print_trainable_parameters()

losses = train(model, loader, epochs=2, val_loader=val_loader, save_dir=ADAPTERS['attention_mlp_0.5B'])

del model
free_memory()

### 2.3. QLoRA на 1.5B, внимание + MLP (только NVIDIA GPU)

In [ ]:
model = make_qlora_model(qlora_model_name, ATTENTION_MLP)
model.print_trainable_parameters()

losses = train(model, loader, epochs=2, val_loader=val_loader, save_dir=ADAPTERS['qlora_1.5B'])

del model
free_memory()

## 3. Оценка готовых моделей

Каждая модель загружается из адаптера, прогоняется по всем тестам, а результаты сохраняются в `results/<модель>__<тест>.json`. Чтобы добавить модель, допишите её в `ADAPTERS` и в `LOADERS`.

QLoRA-адаптер подключается к 4-битной базе на NVIDIA GPU (как при обучении) и к fp16-базе на остальных устройствах.

In [ ]:
LOADERS = {
    'attention_0.5B': lambda: load_lora_model(ADAPTERS['attention_0.5B']),
    'attention_mlp_0.5B': lambda: load_lora_model(ADAPTERS['attention_mlp_0.5B']),
    'qlora_1.5B': lambda: load_qlora_model(ADAPTERS['qlora_1.5B'], quantized=torch.cuda.is_available()),
}

for tag, load in LOADERS.items():
    model = load()
    for test_name, test_set in TESTS.items():
        results = run_all(test_set, model, save_path=f'{RESULTS_DIR}/{tag}__{test_name}.json')
        print(tag, test_name, metrics(results))
    del model
    free_memory()

## 4. Сводка

Читает сохранённые результаты, поэтому GPU и модели не нужны.

In [ ]:
print(f"{'модель__тест':32} {'n':>4} {'valid':>6} {'exact':>6} {'fields':>7}")
for path in sorted(glob.glob(f'{RESULTS_DIR}/*.json')):
    m = metrics(load_results(path))
    name = os.path.basename(path)[:-5]
    print(f"{name:32} {m['n']:>4} {m['valid_rate']:>6} {m['exact_match']:>6} {m['field_acc']:>7}")

## 5. Разбор ошибок

Какие поля ошибаются чаще всего и на каких объявлениях.

In [ ]:
tag, test_name = 'attention_mlp_0.5B', 'real_2'          # что разбираем
results = load_results(f'{RESULTS_DIR}/{tag}__{test_name}.json')

print(metrics(results))
print(field_errors(results).most_common())

In [ ]:
for r in results:
    if r['status'] == 'ok' and r['match'] != 9:
        diff = {k: (r['parsed'][k], r['expected'][k]) for k in r['expected'] if r['parsed'][k] != r['expected'][k]}
        print(diff, '|', r['text'][:100].replace('\n', ' '))
        print()

## 6. Проверка забывания

Обычные вопросы без нашего промпта: ответы модели с адаптером и без него (`disable_adapter` временно отключает адаптер).

In [ ]:
model = load_lora_model(ADAPTERS['attention_mlp_0.5B'])

questions = [
    "Объясни, что такое нейронная сеть, в двух предложениях.",
    "Чем кошка отличается от собаки?",
    "Чем список отличается от кортежа в Python?",
    "Напиши функцию на Python, которая проверяет, является ли строка палиндромом.",
    "Напиши короткое объявление о сдаче двухкомнатной квартиры.",
    "Сколько будет 17 умножить на 23?",
]

for q in questions:
    with model.disable_adapter():
        base_answer = chat(model, q)
    tuned_answer = chat(model, q)
    print('ВОПРОС:', q)
    print('--- без адаптера:', base_answer)
    print('--- с адаптером:', tuned_answer)
    print()

## 7. Свой текст

In [ ]:
text = 'Сдам 2к квартиру, 54 кв.м, 5/9 этаж, м. Сокольники, 65 тыс. в месяц. Без животных. Звонить +7 916 123-45-67, Ирина'
print(generate_json(model, text))

del model
free_memory()

## 8. Деплой: GGUF и Ollama

Цепочка: слияние адаптера с базой (fp32) → GGUF F16 → квантизация (Q8_0, Q4_K_M) → модель в Ollama. Промежуточные файлы удаляются.

Нужно заранее: `llama.cpp` в `~/llama.cpp` (из него берутся скрипт конвертации и собранная утилита `llama-quantize`, путь задан в `config.py`) и запущенный Ollama.

Важно: QLoRA-адаптер обучался на 4-битной базе, а сливается с базой в полной точности. На непривычных (реальных) текстах слитая модель немного отличается от обученной: на реальных объявлениях `valid` около 0,93 против 1,00, `exact` и `field_acc` в пределах нескольких пунктов. Подробности в README.

### 8.1. Экспорт

In [ ]:
export_gguf(ADAPTERS['qlora_1.5B'], 'listing')      # создаст gguf/listing-q8_0.gguf, gguf/listing-q4_k_m.gguf и модели в Ollama

### 8.2. Оценка моделей из Ollama

In [ ]:
for tag in ['listing-q8_0', 'listing-q4_k_m']:
    for test_name, test_set in TESTS.items():
        results = run_all(test_set, tag, generate=ollama_generate)
        print(tag, test_name, metrics(results))

### 8.3. Свой текст

In [ ]:
print(ollama_generate('listing-q8_0', text))